# Synthetic Ride-Hailing Price Analysis

Notebook nay cap nhat theo `Synthetic_plan.md`: tao va phan tich synthetic data cho gia dat xe cong nghe. Trong tam la mo hinh gia: rain, rush hour, location effect va driver effect. Dataset khong con mo hinh `driver_accept`; phan do co the tach thanh giai doan sau.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/driver_synthetic_trips.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["trip_time"])
df.head()

## 1. Kiem tra cau truc dataset

In [ ]:
display(df.shape)
display(df.dtypes)
display(df.describe(include="all"))

In [ ]:
group_counts = (
    df.groupby(["driver_id", "rain", "rush_hour"], observed=True)
    .size()
    .rename("n_trips")
    .reset_index()
)
min_group_count = group_counts["n_trips"].min()
summary = {
    "n_trips": len(df),
    "n_drivers": df["driver_id"].nunique(),
    "n_origin_zones": df["origin_zone"].nunique(),
    "n_destination_zones": df["destination_zone"].nunique(),
    "rain_rate": df["rain"].mean(),
    "rush_hour_rate": df["rush_hour"].mean(),
    "min_driver_weather_time_group_count": min_group_count,
}
summary

## 2. Fitted values cua mo hinh co so

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(df[["rain", "rush_hour"]], df["delta_price"])

baseline_coefficients = pd.DataFrame(
    {
        "term": ["beta0", "beta1_rain", "beta2_rush_hour"],
        "estimate": [baseline_model.intercept_, *baseline_model.coef_],
        "true_value": [2000, 8000, 12000],
    }
)
baseline_coefficients

In [ ]:
fitted_groups = pd.DataFrame(
    {
        "rain": [0, 1, 0, 1],
        "rush_hour": [0, 0, 1, 1],
    }
)
fitted_groups["fitted_delta_price"] = baseline_model.predict(fitted_groups[["rain", "rush_hour"]])
fitted_groups["group"] = [
    "Khong mua - binh thuong",
    "Mua - binh thuong",
    "Khong mua - cao diem",
    "Mua - cao diem",
]
fitted_groups

In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(data=fitted_groups, x="group", y="fitted_delta_price", hue="group", legend=False)
plt.xticks(rotation=20, ha="right")
plt.title("Bon fitted values cua mo hinh co so")
plt.xlabel("")
plt.ylabel("Fitted delta price")
plt.tight_layout()

## 3. So sanh M0, M1, M2 tren train-test split theo thoi gian

In [ ]:
def build_design_frame(frame, features):
    return pd.get_dummies(frame[features], drop_first=True, dtype=float)


def adjusted_r2(r2, n_rows, n_features):
    return 1 - (1 - r2) * (n_rows - 1) / (n_rows - n_features - 1)


def aic_bic(y_true, y_pred, n_features):
    residual = y_true - y_pred
    rss = float(np.sum(residual ** 2))
    n_rows = len(y_true)
    rss_per_row = max(rss / n_rows, 1e-9)
    aic = n_rows * np.log(rss_per_row) + 2 * n_features
    bic = n_rows * np.log(rss_per_row) + np.log(n_rows) * n_features
    return aic, bic


def fit_and_evaluate(train, test, features, model_name):
    train_x = build_design_frame(train, features)
    test_x = build_design_frame(test, features).reindex(columns=train_x.columns, fill_value=0)

    model = LinearRegression()
    model.fit(train_x, train["delta_price"])
    predicted = model.predict(test_x)

    mae = mean_absolute_error(test["delta_price"], predicted)
    rmse = mean_squared_error(test["delta_price"], predicted) ** 0.5
    r2 = r2_score(test["delta_price"], predicted)
    adj_r2 = adjusted_r2(r2, len(test), train_x.shape[1])
    aic, bic = aic_bic(test["delta_price"].to_numpy(), predicted, train_x.shape[1] + 1)

    return model, train_x.columns, pd.Series(
        {
            "model": model_name,
            "n_features": train_x.shape[1],
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "adjusted_R2": adj_r2,
            "AIC": aic,
            "BIC": bic,
        }
    ), predicted


df_sorted = df.sort_values("trip_time").reset_index(drop=True)
split_index = int(len(df_sorted) * 0.8)
train = df_sorted.iloc[:split_index]
test = df_sorted.iloc[split_index:]

models = {
    "M0_rain_rush": ["rain", "rush_hour"],
    "M1_location": ["rain", "rush_hour", "origin_zone", "destination_zone"],
    "M2_driver_location": ["rain", "rush_hour", "origin_zone", "destination_zone", "driver_id"],
}

fitted = {}
metrics = []
predictions = {}
for model_name, features in models.items():
    model, columns, metric_row, predicted = fit_and_evaluate(train, test, features, model_name)
    fitted[model_name] = {"model": model, "columns": columns, "features": features}
    metrics.append(metric_row)
    predictions[model_name] = predicted

metrics_df = pd.DataFrame(metrics)
metrics_df

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=metrics_df, x="model", y="RMSE", hue="model", legend=False)
plt.title("So sanh RMSE giua M0, M1, M2")
plt.xlabel("")
plt.ylabel("RMSE")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()

## 4. Actual versus predicted

In [ ]:
actual_vs_predicted = test[["trip_id", "delta_price"]].copy()
actual_vs_predicted["predicted_delta_price"] = predictions["M2_driver_location"]
actual_vs_predicted.head()

In [ ]:
plt.figure(figsize=(5, 5))
sns.scatterplot(data=actual_vs_predicted, x="predicted_delta_price", y="delta_price", alpha=0.45)
min_value = min(actual_vs_predicted["predicted_delta_price"].min(), actual_vs_predicted["delta_price"].min())
max_value = max(actual_vs_predicted["predicted_delta_price"].max(), actual_vs_predicted["delta_price"].max())
plt.plot([min_value, max_value], [min_value, max_value], color="red", linestyle="--", label="y = x")
plt.title("Actual versus predicted delta price - M2")
plt.xlabel("Predicted delta price")
plt.ylabel("Actual delta price")
plt.legend()
plt.tight_layout()

## 5. Scatter plot voi feature lien tuc distance

In [ ]:
distance_model = LinearRegression()
distance_features = ["rain", "rush_hour", "distance_km"]
distance_model.fit(df[distance_features], df["delta_price"])

line_x = np.linspace(df["distance_km"].min(), df["distance_km"].max(), 100)
line_frame = pd.DataFrame({"rain": 0, "rush_hour": 0, "distance_km": line_x})
line_y = distance_model.predict(line_frame)

plt.figure(figsize=(8, 4))
sample = df.sample(n=900, random_state=42)
sns.scatterplot(data=sample, x="distance_km", y="delta_price", alpha=0.25)
plt.plot(line_x, line_y, color="red", label="Fitted line khi rain=0, rush_hour=0")
plt.title("Delta price theo distance_km")
plt.xlabel("Distance km")
plt.ylabel("Delta price")
plt.legend()
plt.tight_layout()

## 6. Ket luan ngan

In [ ]:
beta0 = baseline_coefficients.loc[baseline_coefficients["term"] == "beta0", "estimate"].iloc[0]
beta1 = baseline_coefficients.loc[baseline_coefficients["term"] == "beta1_rain", "estimate"].iloc[0]
beta2 = baseline_coefficients.loc[baseline_coefficients["term"] == "beta2_rush_hour", "estimate"].iloc[0]
best_model = metrics_df.sort_values("RMSE").iloc[0]

conclusion = f"""
Ket luan:
- Mo hinh co so uoc luong beta0 = {beta0:,.0f}, beta1_rain = {beta1:,.0f}, beta2_rush_hour = {beta2:,.0f}.
- Cac gia tri nay gan voi he so synthetic that: 2,000; 8,000; 12,000.
- M1 them origin_zone va destination_zone de kiem tra anh huong khu vuc.
- M2 them driver_id de uoc luong driver fixed effects.
- Theo RMSE tren test set, mo hinh tot nhat la {best_model['model']} voi RMSE = {best_model['RMSE']:,.0f}.
- Vi rain va rush_hour la bien nhi phan, mo hinh co so tao bon fitted values thay vi mot duong hoi quy lien tuc.
"""
print(conclusion)